# MVP — Machine Learning & Analytics

**Nome:**             _Lucas Davi Santos Farias_  
**Matrícula:**        _4052025002520_  
**Data:**             _01/06/2026_  
**Dataset:**          _[Brain Tumor MRI](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)_  
**Tipo de problema:** _Classificação_  

# 1. Definição do problema

## 1.1 Descrição do problema

O problema é de classificação de imagens de ressonâncias magnéticas cerebrais, com objetivo de identificar se o cérebro da imagem possui algum tipo de câncer ou não. Esse tipo de problema é um exemplo de como a área de ciências de dados e machine learning pode auxiliar profissionais de outras áreas com ferramentas de auxílio ao diagnóstico. Nesse caso, o modelo seria capaz de ajudar o profissional da saúde a identificar um câncer com certa acurácia.

## 1.2 Objetivos do MVP

Objetivo principal do trabalho:

- Utilizar CNNs para classificar uma imagem de ressonância magnética em uma das 4 possíveis classes, comparando uma rede neural profunda hiperconectada como baseline com CNNs cada vez mais avançandas, levando em consideração o custo de criação do modelo e seu desempenho.

Obejetivos secundários:

- Criação de GAN para gerar imagens artificiais de ressonância magnética de uma ou mais classes;
- Utilização de segmentação de dados para identificar o local da lesão no cérebro.

## 1.3 Tipo de problema

**Tipo escolhido:** _Classificação_  
**Justificativa:** _O problema consiste em classificar uma imagem em uma das quatro classes possíveis_

## 1.4 Critérios de sucesso

Usando como referência o artigo [A guide to interpreting the area under the curve value](https://pmc.ncbi.nlm.nih.gov/articles/PMC10664195/#abstract1), disponibilizado no National Library of Medicine que define algumas métricas para compreender se um modelo desenvolvido para diagnósticos de pacientes é bom o suficiente.

Será utilizado no mvp os seguintes critérios:

- Usar AUC com objetivo de obter valor mínimo de 0,8 - considerado bom pelo artigo - com um intervalo de confiança estreito, garantindo bons resultados;
- Utilização de matriz de confusão.

Para avaliar se um modelo é melhor do que outro, o artigo recomenda a utilização do teste De-Long, que compara duas curvas ROC, avaliando se a diferença de performance entre modelos é estatisticamente significante.

# 2. Ambiente, bibliotecas e reprodutibilidade

In [1]:
import random
import warnings
import pathlib
import os
import zipfile
import io

import PIL
import PIL.Image
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

warnings.filterwarnings("ignore")

SEED = 67  # Para garantir reprodutibilidade do código
np.random.seed(SEED)
random.seed(SEED)

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

# 3. Seleção e carga dos dados

## 3.1 Fonte dos dados

O dataset escolhido foi o [Brain Tumor MRI](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

Esse problema foi escolhido pelos seguintes motivos:

- Possibilidade de demonstrar o poder do aprendizado de máquina a leigos; mostrando a capacidade de ensinar um modelo a identificar algo que apenas um profissional treinado conseguiria. As imagens do dataset e o output do modelo são facilmente compreendidos e causam impacto;
- A utilização de modelos CNNs que durante o curso foi o que mais me deu curiosidade para aplicar;
- A possibilidade de aplicar outras técnicas como GANs para criação de imagens artificiais de cérebros com algum tipo de câncer; Será que um profissional iria perceber que a imagem é artificial?
- A possibilidade de aplicar segmentação de imagem para identificar em qual local do cérebro existe uma lesão;

O dataset possui a licença Attribution 4.0 International, permitindo compartilhamento e modificação.


## 3.2 Carga dos dados

Os dados foram armazenados no Google Driver em link público no formato ZIP.

In [2]:
classes = ("glioma", "meningioma", "pituitary", "notumor")

In [3]:
dataset_folder = pathlib.Path('/content/brain_mri_dataset')
for classe in classes:
    try:
        os.mkdir(dataset_folder/classe)
    except FileExistsError:
        continue

In [ ]:
github_url = "https://raw.githubusercontent.com/LucasDSF7/Machine-Learning-Puc-Rio/refs/heads/main/mvp/brain_mri_data/"

with requests.Session() as session:
    for classe in classes:
        for i in range(3):
            r = session.get(github_url+f"{classe}{i}.zip")
            with zipfile.ZipFile(io.BytesIO(r.content), "r") as zipped:
                for name in zipped.namelist():
                    zipped.extractall(dataset_folder/classe)

KeyError: "There is no item named 'Te-gl_1.jpg' in the archive"

In [7]:
image_count = len(list(dataset_folder.glob('*/*/*.jpg')))
print(image_count)

7200


In [6]:
os.listdir("/content/brain_mri_dataset/pituitary")

['pituitary0', 'pituitary2', 'pituitary1']